# Fine-tuned VideoMAE — post-training analysis

This notebook reads Hugging Face `Trainer` logs under a run directory (e.g. `checkpoints/videomae-workout`), plots learning curves, and (optionally) runs test-set **F1** and **inference timing** using the same data path as `scripts/eval.py`.

**Configure** `RUN_DIR` below to point at your checkpoint folder (the directory that contains `config.json` and `checkpoint-*` subfolders).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Repo root (parent of this notebook's folder)
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "analysis").is_dir() and (REPO_ROOT / "utils").is_dir():
    pass
elif (REPO_ROOT.parent / "utils").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from analysis.training_history import (
    load_train_results_json,
    load_training_history,
    moving_average,
    resolve_trainer_state_path,
)

# VideoMAE YouTube run (change as needed)
RUN_DIR = REPO_ROOT / "checkpoints" / "videomae-youtube" # change here for the model to be analyzed
print("Run directory:", RUN_DIR.resolve())

## 1. Training vs validation loss (convergence)

`trainer_state.json` stores `log_history`: training rows (step, `loss`, `learning_rate`, `grad_norm`) and validation rows (`eval_loss`, `eval_accuracy`, …).

In [ ]:
ts_path = resolve_trainer_state_path(RUN_DIR)
history = load_training_history(trainer_state_path=ts_path)

train_df = pd.DataFrame(history.train)
eval_df = pd.DataFrame(history.eval)

fig, ax = plt.subplots(figsize=(10, 5))
if not train_df.empty and "loss" in train_df.columns:
    ax.plot(train_df["step"], train_df["loss"], alpha=0.35, label="train loss (raw)")
    if len(train_df) > 5:
        sm = moving_average(train_df["loss"].tolist(), window=max(5, len(train_df) // 50))
        ax.plot(train_df["step"], sm, linewidth=2, label="train loss (moving avg)")
if not eval_df.empty and "eval_loss" in eval_df.columns:
    ax.plot(eval_df["step"], eval_df["eval_loss"], "o-", markersize=4, label="validation loss")
ax.set_xlabel("global step")
ax.set_ylabel("loss")
ax.set_title("Training and validation loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(json.dumps(history.trainer_meta, indent=2))

## 2. Test set: F1 and inference speed

Uses `analysis.inference_metrics.evaluate_checkpoint_on_test` — same logic as `scripts/eval.py`. Default **`data/test/test/<class_name>/`** must match the training layout (one folder per exercise). If that tree yields no labeled videos, the code **falls back** to the stratified test split from `data_roots` (often ~256 clips — **not** your small `data/test` folder). Restart the kernel after `git pull` so `evaluate_checkpoint_on_test` matches the repo (avoid stale `repo_root` errors).

Default **batch size 1** runs clips **one at a time** with a **tqdm** bar (`sample` unit); increase `batch_size` for faster runs (`batch` unit). First batch(es) are warm-up and excluded from timing.

Paths in config are resolved to the **repository root** (not Jupyter’s cwd). After `git pull` or editing analysis code, **Restart kernel** so imports reload.

In [ ]:
from analysis.inference_metrics import evaluate_checkpoint_on_test

# Saved model directory (same as: python scripts/eval.py --checkpoint …)
CHECKPOINT = RUN_DIR

# Default batch_size=1: one clip per tqdm step (unit: sample). Use batch_size=4+ for speed (unit: batch).
result = evaluate_checkpoint_on_test(
    checkpoint=CHECKPOINT,
    batch_size=1,
    warmup_batches=2,
    show_progress=True,
)
print(f"Test samples: {result.n_samples}")
print(f"Accuracy:     {result.accuracy:.4f}")
print(f"F1 macro:     {result.f1_macro:.4f}")
print(f"F1 weighted:  {result.f1_weighted:.4f}")
print(f"Wall ms/sample (inference only): {result.wall_time_per_sample_s * 1000:.2f}")
print(f"Throughput:   {result.throughput_samples_per_s:.3f} samples/s")
print("\n" + result.classification_report)

## 3. Other useful signals from the logs

- **Learning rate** schedule vs step (warmup + decay are visible).
- **Gradient norm** spikes can flag instability or difficult batches.
- **Validation accuracy** alongside loss shows whether the metric tracked during training improved in sync with loss.
- **`train_results.json`** summarizes end-of-run throughput (train) — compare with eval throughput in `log_history` for val-speed trends.

In [ ]:
train_results = load_train_results_json(RUN_DIR)
if train_results:
    print("train_results.json:", json.dumps(train_results, indent=2))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
if not train_df.empty and "learning_rate" in train_df.columns:
    axes[0].plot(train_df["step"], train_df["learning_rate"], color="C2")
    axes[0].set_ylabel("learning rate")
    axes[0].set_title("LR schedule")
    axes[0].grid(True, alpha=0.3)
if not train_df.empty and "grad_norm" in train_df.columns:
    axes[1].semilogy(train_df["step"], train_df["grad_norm"], alpha=0.7)
    axes[1].set_ylabel("grad norm (log)")
    axes[1].set_xlabel("global step")
    axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if not eval_df.empty and "eval_accuracy" in eval_df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(eval_df["step"], eval_df["eval_accuracy"], "s-", label="val accuracy")
    ax.set_xlabel("global step")
    ax.set_ylabel("accuracy")
    ax.set_title("Validation accuracy (Trainer eval)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

# Train vs val gap: align last train loss at or before each eval step
if not train_df.empty and not eval_df.empty and "eval_loss" in eval_df.columns:
    t = train_df.sort_values("step")[["step", "loss"]].rename(columns={"loss": "train_loss"})
    e = eval_df.sort_values("step")[["step", "eval_loss"]]
    gap_df = pd.merge_asof(e, t, on="step", direction="backward")
    gap_df["gap"] = gap_df["eval_loss"] - gap_df["train_loss"]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(gap_df["step"], gap_df["gap"], label="val_loss - train_loss (asof backward)")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("global step")
    ax.set_ylabel("loss gap")
    ax.set_title("Train–validation loss gap")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()